# Bangladesh: 0212_lake_area_precipitation

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0212_lake_area_precipitation
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = Path(os.environ.get('BDGEE_OUTPUT_DIR', _REPO_ROOT / 'outputs' / 'water' / '0212_lake_area_precipitation'))
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""00011_lake_area_precipitation.ipynb

Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Google Earth Engine Tutotrial-200: Lake Area and Precipoitation Correlation Analysis in Python
This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor

"""
import ee
None
None
'# **Study Area**'
import geemap
map = geemap.Map(basemap='SATELLITE')
map
roi = BD_ROI
roi
'# **Landsat Collections**'
landsat8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2022', '2026').filterBounds(roi).filter(ee.Filter.lt('CLOUD_COVER', 45))
landsat9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2').filterDate('2022', '2026').filterBounds(roi).filter(ee.Filter.lt('CLOUD_COVER', 45))
landsat = landsat8.merge(landsat9).sort('system:time_start')

def ndwi(img):
    qa_band = img.select('QA_PIXEL')
    bit1 = qa_band.bitwiseAnd(1 << 1).neq(0)
    bit2 = qa_band.bitwiseAnd(1 << 2).neq(0)
    bit3 = qa_band.bitwiseAnd(1 << 3).neq(0)
    bit4 = qa_band.bitwiseAnd(1 << 4).neq(0)
    mask = bit1.Or(bit2).Or(bit3).Or(bit4)
    sr_bands = img.select('SR_B.*').multiply(2.75e-05).add(-0.2)
    index = sr_bands.normalizedDifference(['SR_B3', 'SR_B5']).rename('ndwi')
    return index.updateMask(mask.Not()).copyProperties(img, ['system:time_start'])
landsat_ndwi = landsat.map(ndwi)
landsat_ndwi
'# **to Xarray**'
from xee import helpers
from shapely.geometry import shape
import xarray as xr
landsat8.first().select('SR_B5').projection()
grid_params = helpers.fit_geometry(geometry=shape(roi.getInfo()), grid_crs='EPSG:32639', grid_scale=(100, -100))
ds = xr.open_dataset(landsat_ndwi, engine='ee', **grid_params)
ds = ds.sortby('time') * 1
'# **Landsat Resampling**'
ds_monthly = ds.resample(time='M').median('time')
ds_monthly.isel(time=30).ndwi.plot(x='x', y='y', robust=True)
'# **Lake Area**'
import rioxarray
x_res, y_res = ds_monthly.rio.resolution()
y_res
pixel_area = abs(x_res * y_res)
pixel_area
lake = (ds_monthly >= 0.2).astype(int)
lake_pixels = (lake.ndwi == 1).astype(int).sum(dim=['y', 'x'])
lake_area = (lake_pixels * pixel_area / 1000000.0).to_dataframe()
lake_area.plot()
lake.ndwi.isel(time=11).plot(x='x', y='y')
'# **Precipitation Collection**'
pr = ee.ImageCollection('NOAA/PERSIANN-CDR').filterDate('2022', '2026')
pr
'# **Pr to Xarray**'
pr_grid = helpers.fit_geometry(geometry=shape(roi.getInfo()), grid_crs='EPSG:4326', grid_scale=(0.27, -0.27))
ds_pr = xr.open_dataset(pr, engine='ee', **pr_grid)
ds_pr = ds_pr.sortby('time') * 1
'# **Pr Monthly**'
pr_monthly = ds_pr.resample(time='M').sum('time')
pr_monthly_mean = pr_monthly.mean(dim=['x', 'y']).to_dataframe()
pr_monthly_mean.plot()
'# **Correlation**'
import pandas as pd
result = pd.merge(lake_area, pr_monthly_mean, left_index=True, right_index=True)
result
result.corr()
result.corr().iloc[0, 1] ** 2


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'water', '0212_lake_area_precipitation', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'water')  # explicit upload; requires Drive OAuth scope
